# autosearch diag — Phase 2 calibration diagnosis (GPU T4 x2)

Bounded diagnostic run: verifies the calibration root-cause fixes on real
T4 hardware.

- Bug A: LD_LIBRARY_PATH forwarded by the runner's stripped env
- Bug B: time_budget_s forwarded to train.py (AUTORESEARCH_TIME_BUDGET)
- Bug C: calibration failures retain per-GPU diagnostics

Expected wall time: ~15-25 min. Secrets: loads GEMINI_API_KEY via Kaggle
Secrets; never printed.

In [ ]:
# 1. Clone the project at origin/main, show identity, install deps.
import os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working')
PROJ = WORK / 'autosearch_kaggle'
os.chdir(WORK)  # never rmtree the directory the kernel is sitting in
if PROJ.exists():
    import shutil; shutil.rmtree(PROJ)
subprocess.run(['git', 'clone', '-q',
                'https://github.com/MokshonWork/autosearch_kaggle.git',
                str(PROJ)], check=True)
os.chdir(PROJ)
sys.path.insert(0, str(PROJ))
print(subprocess.run(['git', 'log', '--oneline', '-4'],
                     capture_output=True, text=True).stdout)
print(subprocess.run(['nvidia-smi', '--list-gpus'],
                     capture_output=True, text=True).stdout)
print('python', sys.version)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'rustbpe>=0.1.0', 'tiktoken>=0.11.0'], check=True)
import torch
print('torch', torch.__version__, 'cuda', torch.version.cuda,
      'available', torch.cuda.is_available(), 'count', torch.cuda.device_count())

In [ ]:
# 2. Secret -> env only (never printed, never committed).
# Phase-2 diagnostics make NO Gemini API calls, so a missing secret must not
# block calibration diagnosis; the Phase-3 smoke kernel still hard-requires it.
#
# NOTE: we do NOT use kaggle_secrets.UserSecretsClient. Kaggle injects
# KAGGLE_URL_BASE=https://api.kaggle.com, but the UserSecretsService endpoint
# is served only by https://www.kaggle.com; against the injected host every
# request gets HTTP 400 with an empty body, which kaggle_web_client flattens
# into a generic ConnectionError. src/kaggle_secret.py hits the working host
# and separates "not attached to this kernel" from real backend failures.
import os, sys
sys.path.insert(0, '/kaggle/working/autosearch_kaggle')
from src.kaggle_secret import (load_api_key, SecretNotAttachedError,
                               SecretAuthError, SecretBackendError)
secret_ok = False
try:
    load_api_key('GEMINI_API_KEY')
    secret_ok = bool(os.environ.get('GEMINI_API_KEY'))
except SecretNotAttachedError as exc:
    print('SECRET NOT ATTACHED:', exc)
    print('continuing WITHOUT the secret (no API calls in this kernel)')
except (SecretAuthError, SecretBackendError) as exc:
    print('SECRET FETCH FAILED:', type(exc).__name__, exc)
    print('continuing WITHOUT the secret (no API calls in this kernel)')
print('GEMINI_API_KEY loaded:', secret_ok)

In [ ]:
# 3. Nested training repo needs its own .git (bug D guard) + data prep.
import subprocess, sys
from pathlib import Path
PROJ = Path('/kaggle/working/autosearch_kaggle')
repo = PROJ / 'repo'
if not (repo / '.git').exists():
    for a in (['init', '-q'], ['add', '-A'],
              ['-c', 'user.email=k@kaggle', '-c', 'user.name=k',
               'commit', '-q', '-m', 't4-port baseline']):
        subprocess.run(['git', '-C', str(repo), *a], check=True)
    print('repo/.git re-initialized')
print('inner HEAD:', subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'],
    text=True).strip())
print('inner git root:', subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', '--show-toplevel'],
    text=True).strip())
r = subprocess.run([sys.executable, 'prepare.py', '--num-shards', '4'],
                   cwd=str(repo), capture_output=True, text=True, timeout=3600)
print(r.stdout[-1500:])
if r.stderr:
    print('STDERR:', r.stderr[-800:])
print('prepare exit:', r.returncode)
assert r.returncode == 0, 'dataset/tokenizer prep failed'

In [ ]:
# 4. Bug A: CUDA inside the runner's restricted child env, per GPU.
import os, subprocess, sys
from src.tools.runner import stripped_env
probe = ("import os,torch; "
         "print('available:',torch.cuda.is_available()); "
         "print('count:',torch.cuda.device_count()); "
         "print('name:', torch.cuda.get_device_name(0) "
         "      if torch.cuda.is_available() else 'none')")
restricted = {}
for gpu in (0, 1):
    env = stripped_env(gpu)
    print(f"gpu{gpu}: LD_LIBRARY_PATH forwarded = "
          f"{'LD_LIBRARY_PATH' in env}")
    p = subprocess.run([sys.executable, '-c', probe], env=env,
                       capture_output=True, text=True, timeout=120)
    print(f'--- restricted child gpu{gpu} exit={p.returncode} ---')
    print(p.stdout.strip())
    if p.stderr.strip():
        print('stderr:', p.stderr[-600:])
    restricted[gpu] = (p.returncode, 'available: True' in p.stdout)
assert all(ok for _, ok in restricted.values()), (
    f'CUDA not visible in restricted child: {restricted}')

In [ ]:
# 5. Three-step CUDA preflight through the real runner, per GPU.
from pathlib import Path
from src.tools.runner import run_training
repo = Path('/kaggle/working/autosearch_kaggle/repo').resolve()
preflight = {}
for gpu in (0, 1):
    r = run_training(repo, gpu, f'PREFLIGHT-GPU-{gpu}',
                     time_budget_s=300, startup_allowance_s=240,
                     preflight_steps=3)
    preflight[gpu] = r
    print(f'gpu{gpu}: status={r.status} wall={r.wall_seconds}s')
    print(r.stdout_tail[-600:])
    print('======')
assert all(r.status == 'ok' and 'PREFLIGHT_OK' in r.stdout_tail
           for r in preflight.values()), 'preflight failed'

In [ ]:
# 6. Bug B: full bounded runs (10s budget) on both GPUs, concurrently.
# Must finish WELL under the 250s hard timeout with a finite val_bpb.
from concurrent.futures import ThreadPoolExecutor
from src.tools.runner import run_training
import math

def bounded(gpu):
    return run_training(repo, gpu, f'BUDGET10-GPU-{gpu}',
                        time_budget_s=10, startup_allowance_s=240)

with ThreadPoolExecutor(max_workers=2) as ex:
    bounded_rs = list(ex.map(bounded, [0, 1]))

for r in bounded_rs:
    print(f'gpu{r.gpu}: status={r.status} val_bpb={r.val_bpb} '
          f'steps={r.steps} wall={r.wall_seconds}s')
    print(r.stdout_tail[-1500:])
    print('======')

for r in bounded_rs:
    assert r.status == 'ok', f'gpu{r.gpu} bounded run failed: {r.status}'
    assert (r.val_bpb is not None and math.isfinite(r.val_bpb)
            and r.val_bpb > 0), f'gpu{r.gpu} produced no valid val_bpb'
    assert r.wall_seconds < 250, f'gpu{r.gpu} hit the hard timeout region'


In [ ]:
# 7. Bug C: the REAL calibration path (orchestrator + config.smoke.yaml).
# With fixes: epsilon + baseline from 1 concurrent pair. Without: the
# RuntimeError now carries per-GPU status/wall/tail diagnostics.
import os
if not os.environ.get('GEMINI_API_KEY'):
    os.environ['GEMINI_API_KEY'] = 'diag-placeholder-not-a-key'
    print('using placeholder key (calibration makes no API calls)')
from src.config import load_config
from src.llm.client import BudgetTracker, GeminiClient
from src.orchestrator import Orchestrator
cfg = load_config('config.smoke.yaml')
cfg.paths.work_dir = '/kaggle/working/diag_work'
cfg.paths.memory_dir = '/kaggle/working/diag_memory'
cfg.paths.stop_file = '/kaggle/working/diag_STOP'
budget = BudgetTracker(cfg.budget.max_api_calls_total,
                       cfg.budget.max_tokens_total,
                       cfg.budget.max_calls_per_round)
client = GeminiClient(cfg.llm, budget)  # validates the key; no API call
orch = Orchestrator(cfg, client)
calib_ok, calib_detail = False, ''
try:
    orch._calibrate_if_needed()
    calib_ok = True
    calib_detail = (f"epsilon={orch.state['epsilon']} "
                    f"baseline={orch.state['baseline_bpb']} "
                    f"values={orch.state['calibration_values']}")
    print('CALIBRATION OK', calib_detail)
except Exception as e:
    calib_detail = f'{type(e).__name__}: {e}'
    print('CALIBRATION FAILED')
    print(calib_detail[:4000])
summary_path = Path('/kaggle/working/diag_summary.json')
import json
summary_path.write_text(json.dumps({
    'preflight': {str(g): r.status for g, r in preflight.items()},
    'bounded': {str(r.gpu): {'status': r.status, 'val_bpb': r.val_bpb,
                              'steps': r.steps, 'wall': r.wall_seconds}
                for r in bounded_rs},
    'secret_ok': secret_ok,
    'calibration_ok': calib_ok,
    'calibration_detail': calib_detail,
}, indent=2))
print('summary written to', summary_path)
assert calib_ok, 'calibration failed — see diagnostics above'
print('DIAG_PHASE2_COMPLETE')